# breast cancer one-class svm, with probabilities

four ways to turn the svm score into a number from 0 to 1. i want to see which one make sense when someone ask me what 0.83 mean.

In [ ]:
# cac thu vien can thiet
import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar, brentq
from scipy.special import expit
from scipy.stats import gamma
from sklearn.datasets import load_breast_cancer
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)  # de chay lai cho ket qua giong nhau

In [ ]:
# load du lieu ung thu vu cua sklearn
bc = load_breast_cancer()
Xall, yall = bc.data, bc.target
# yall = 1 nghia la lanh tinh (khong ung thu), 0 la ac tinh

# tach: chi train tren mau lanh tinh
normal_idx = np.where(yall == 1)[0]
anomaly_idx = np.where(yall == 0)[0]
rng.shuffle(normal_idx)

# 60% mau lanh tinh dem fit svm, con lai gop voi mau ac tinh de test
cut = int(len(normal_idx) * 0.6)
fit_idx = normal_idx[:cut]
held_normal = normal_idx[cut:]

# chuan hoa feature - fit scaler chi tren tap train
sc = StandardScaler().fit(Xall[fit_idx])
Xfit = sc.transform(Xall[fit_idx])
Xeval = sc.transform(np.vstack([Xall[held_normal], Xall[anomaly_idx]]))
yeval = np.r_[np.ones(len(held_normal)), np.zeros(len(anomaly_idx))]

# in ra: so mau train, so mau test, ti le lanh tinh trong test
len(fit_idx), len(Xeval), yeval.mean().round(3)

In [ ]:
# fit one-class svm tren mau lanh tinh
svm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')
svm.fit(Xfit)

# diem so g(x): duong = giong lanh tinh, am = la
g_fit  = svm.decision_function(Xfit)
g_eval = svm.decision_function(Xeval)
fmax = g_fit.max()  # diem cao nhat trong train, dung sau o gamma scaling

# ve histogram diem so: xanh = lanh tinh, do = ac tinh
plt.figure(figsize=(6.5, 2.6))
for cls, c in [(1, '#2a6'), (0, '#c33')]:
    plt.hist(g_eval[yeval==cls], bins=24, color=c, alpha=0.55, edgecolor='none')
plt.axvline(0, color='k', lw=0.5)  # nguong quyet dinh
plt.xlabel('g(x)')
plt.yticks([])
plt.show()

now the four method. same input (svm score on test set), four outputs.

In [ ]:
# Platt scaling. fit sigmoid p = 1/(1+exp(A*g+B))
# minh khong co nhan that nen lay sign cua svm lam pseudo-label
# bai bao da chi ra cach nay se sup do thanh ham buoc 0-1. minh van lam de so sanh.
from scipy.optimize import minimize

def platt(g_train, g_query, pseudo):
    pseudo = pseudo.astype(float)
    Np = pseudo.sum(); Nn = len(pseudo) - Np
    # target voi smoothing chong overfit (Lin et al 2007)
    t = np.where(pseudo > 0, (Np+1)/(Np+2), 1/(Nn+2))
    def J(ab):
        z = ab[0]*g_train + ab[1]
        return (t*z + np.log1p(np.exp(-z))).sum() + ((1-t)*0).sum()
    out = minimize(J, [0.0, 0.0], method='Nelder-Mead', options={'xatol':1e-5})
    A, B = out.x
    return expit(-(A*g_query + B))

# pseudo-label: svm.predict tra +1 cho lanh tinh, -1 cho la
pseudo = (svm.predict(Xfit) > 0).astype(int)

In [ ]:
# hai cach chia thung (binning) tu bai bao
EPS = 0.001
K = 5  # so thung moi ben

def _nearest_mark(marks, probs, q):
    # gan moi diem query vao mark gan nhat, lay xac suat cua mark do
    j = np.abs(q[:, None] - marks[None, :]).argmin(axis=1)
    return probs[j]

def equidistant(g_train, g_query):
    # chia deu khoang [min, 0] va [0, max]
    a = g_train.min(); b = g_train.max()
    left  = np.linspace(a, 0, K+1)[:-1]
    right = np.linspace(0, b, K+1)[1:]
    marks = np.r_[left, 0.0, right]
    ps = np.linspace(EPS, 1-EPS, marks.size)
    return _nearest_mark(marks, ps, g_query)

def density_bin(g_train, g_query):
    # chia theo quantile (mat do diem), giong cach libsvm 3.3 lam
    qs = (np.arange(K) + 0.5) / K
    neg = g_train[g_train < 0]
    pos = g_train[g_train >= 0]
    nm = np.quantile(neg, qs) if len(neg) else np.zeros(K)
    pm = np.quantile(pos, qs) if len(pos) else np.zeros(K)
    marks = np.r_[nm, 0.0, pm]
    ps = np.linspace(EPS, 1-EPS, marks.size)
    return _nearest_mark(marks, ps, g_query)

In [ ]:
def gamma_scale(g_train, g_query):
    # phuong phap moi cua bai bao: fit phan phoi Gamma cho S = (fmax - g)+
    # roi scale lai sao cho g=0 thi xac suat = 0.5
    S = np.clip(fmax - g_train, 0, None)
    S = S[S > 0]
    m, v = S.mean(), S.var()
    shape, scale = m*m/v, v/m  # moment matching cua Satterthwaite-Welch
    Sq = np.clip(fmax - g_query, 0, None)
    surv = 1 - gamma.cdf(Sq, shape, scale=scale)
    surv0 = 1 - gamma.cdf(fmax, shape, scale=scale)
    # truong hop fmax qua nho thi cong them san de tranh chia cho 0
    surv0 = max(surv0, 1e-9)
    up = 0.5 + 0.5 * (surv - surv0) / max(1 - surv0, 1e-9)
    dn = 0.5 * surv / surv0
    return np.where(surv >= surv0, up, dn).clip(EPS, 1-EPS)

In [ ]:
# tinh xac suat theo ca 4 phuong phap
probs = {
    'platt':       platt(g_fit, g_eval, pseudo),
    'equi-bin':    equidistant(g_fit, g_eval),
    'dens-bin':    density_bin(g_fit, g_eval),
    'gamma-scale': gamma_scale(g_fit, g_eval),
}

# chon 3 benh nhan ngau nhien de xem chi tiet
# cot y: 1 = lanh tinh, 0 = ac tinh
trio = rng.choice(len(g_eval), size=3, replace=False)
print('row    g       y     ' + '   '.join(f'{k:>11}' for k in probs))
for r in trio:
    line = f'{r:>3}  {g_eval[r]:+.3f}  {int(yeval[r])}    '
    line += '   '.join(f'{probs[k][r]:>11.3f}' for k in probs)
    print(line)

In [ ]:
# reliability plot: chia xac suat du doan thanh 10 thung theo quantile
# voi moi thung, tinh xac suat trung binh va ti le lanh tinh thuc te
# duong xam cheo = lien quan hoan hao (du doan dung voi thuc te)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0,1],[0,1], color='#999', lw=0.7)
for name, p in probs.items():
    e = np.quantile(p, np.linspace(0, 1, 11))
    e[0] -= 1e-6; e[-1] += 1e-6
    xs, ys = [], []
    for i in range(10):
        m = (p >= e[i]) & (p < e[i+1])
        if m.sum() >= 2:
            xs.append(p[m].mean()); ys.append(yeval[m].mean())
    ax.plot(xs, ys, '.-', label=name, lw=1)
ax.set_xlabel('predicted'); ax.set_ylabel('empirical')
ax.legend(frameon=False, loc='lower right', fontsize=9)
plt.show()

In [ ]:
# Brier score = MSE giua xac suat du doan va nhan that
# cang nho cang tot
for name, p in probs.items():
    print(f'  {name:<12} {np.mean((p - yeval)**2):.4f}')

TODO try different nu and see if platt stop looking like a step. probly not but worth a check.